In [102]:
from symmer.operators.base import PauliwordOp, QuantumState
import h5py
import numpy as np 
from pyscf import scf
import math

In [103]:
def get_S2(state_mat):
    na = state_mat[0::2].sum()
    nb = state_mat[0::2].sum()
    ab = np.logical_and(state_mat[0::2], state_mat[1::2]).sum()
    a_minus_b_over_2 = (na-nb)/2
    S2_val = a_minus_b_over_2**2 + a_minus_b_over_2 + nb -ab
    return S2_val

In [104]:
chkfile = "O2.h5"

# load chemistry data
mol_load = scf.chkfile.load_mol(chkfile)
scf_load = scf.ROHF(mol_load) 
scf_load.__dict__.update(scf.chkfile.load(chkfile, 'scf')) 

# print chemistry details (note other info can be accessed in these objects see PySCF for further details.)
print(f"n spin orbitals: {2*mol_load.nao}")
print(f"basis: {mol_load.basis}")
xyz_str = mol_load.tostring(format="xyz")
print(f"geometry:\n\n{xyz_str}")
print(f"unit: {mol_load.unit}")
print(f"initaliziation: {scf_load.init_guess}")
E_rohf = scf_load.e_tot
print(f"e_rohf: {E_rohf}")

n spin orbitals: 36
basis: 6-31G
geometry:

2
XYZ from PySCF
O           0.00000000        0.00000000        0.00000000
O           0.00000000        0.00000000        2.25000000
unit: angstrom
initaliziation: minao
e_rohf: -149.44231699065912


In [105]:
serach_space = math.comb(mol_load.nao, mol_load.nelec[0]) * math.comb(mol_load.nao, mol_load.nelec[1]) 
serach_space


1547282880

In [106]:
with h5py.File("O2_data.h5", "r") as infile:
    # print(infile.keys())
    H = PauliwordOp(infile["H_symp"][()], infile["H_coeff"][()])

    hf_psi = QuantumState(infile["hf_state_mat"][()], infile["hf_coeff_vec"][()])
    e_hf   = infile["e_rohf"][()]
    
    cisd_psi = QuantumState(infile["cisd_state_mat"][()], infile["cisd_coeff_vec"][()]).sort()
    e_cisd = infile["e_cisd"][()]

    hci_psi = QuantumState(infile["hci_state_mat"][()], infile["hci_coeff_vec"][()]).sort()
    e_hci = infile["e_hci"][()]

In [107]:
E_hf = H.expval(hf_psi)
E_hf

np.complex128(-149.4423169906593+0j)

In [108]:
print(f"energy ROHF : {E_hf.real:0.9f}")
print(f"energy cisd : {e_cisd:0.9f}")
print(f"energy hci  : {e_hci:0.9f}")

energy ROHF : -149.442316991
energy cisd : -149.590474268
energy hci  : -149.679007153


In [110]:
ref_cisd = QuantumState(cisd_psi.state_matrix[0], [1])
ref_hci = QuantumState(hci_psi.state_matrix[8], [1])

cisd_single_ref_e = H.expval(ref_cisd).real
hci_single_ref_e = H.expval(ref_hci).real

## single reference energies and S^2
print("Single bitstring energies and S^2")
print(f"energy ROHF     : {E_hf.real:0.9f} , S^2: {get_S2(hf_psi.state_matrix[0])}")
print(f"energy cisd ref : {cisd_single_ref_e:0.9f} , S^2: {get_S2(ref_cisd.state_matrix[0])}")
print(f"energy hci ref  : {hci_single_ref_e:0.9f} , S^2: {get_S2(ref_hci.state_matrix[0])}")

Single bitstring energies and S^2
energy ROHF     : -149.442316991 , S^2: 2.0
energy cisd ref : -149.442316991 , S^2: 2.0
energy hci ref  : -148.966547979 , S^2: 2.0


In [116]:
hci_single_ref_e - E_hf.real 

np.float64(0.47576901164583774)

In [111]:
cisd_hf = cisd_psi.dagger * hf_psi
assert cisd_hf.imag == 0


hci_hf = hci_psi.dagger * hf_psi
assert hci_hf.imag == 0

print(f"< CISD | ROHF > = {cisd_hf.real:0.16f}")
print(f"< HCI  | ROHF > = {hci_hf.real:0.16f}")

< CISD | ROHF > = -0.9464588940248271
< HCI  | ROHF > = -0.0936139383549329


In [101]:
overlap_cisd_hf = np.abs(cisd_hf)**2
overlap_hci_hf = np.abs(hci_hf)**2

print(f"|< CISD | ROHF >|^2 = {overlap_cisd_hf:0.16f}")
print(f"|< HCI  | ROHF >|^2 = {overlap_hci_hf:0.16f}")

|< CISD | ROHF >|^2 = 0.8957844380786989
|< HCI  | ROHF >|^2 = 0.0087635694543212


In [114]:
overlap_hci_ref_hci = np.abs( hci_psi.dagger * ref_hci)**2

print(f"|< HCI  | HCI-single >|^2 = {overlap_hci_ref_hci:0.16f}")

|< HCI  | HCI-single >|^2 = 0.0431687057131481


In [115]:
bitstring_hci = ''.join([str(b) for b in ref_hci.state_matrix[0]])
print("base 10 reference state:", int(bitstring_hci,2))

base 10 reference state: 68701388800


In [100]:
Hamiltonian_dictionary = H.to_dictionary
terms = list(Hamiltonian_dictionary.items())[:10] #<-- for space only print first 10 terms (use dictionary for access to all)
for P, coeff in terms:
    print(coeff, P)

(-94.6559847077048+0j) IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(10.719198541746097+0j) ZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(10.719198541746097+0j) IZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(-6.208129397982677e-06+0j) XZXIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(-6.208129397982677e-06+0j) YZYIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(-6.208129397982677e-06+0j) IXZXIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(-6.208129397982677e-06+0j) IYZYIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(0.08513031139488142+0j) XZZZXIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(0.08513031139488142+0j) YZZZYIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(0.08513031139488142+0j) IXZZZXIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
